# 🛡️ Glu-Stock: 01_RESEARCH_SCAN
**Phase**: Universe Selection & Fundamental Filtering

This notebook scans the IDX market, filters for liquidity and high-growth fundamentals, and pushes candidates to the Firebase `research` queue.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib

In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase & Secrets)
import json
import os
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime
from typing import List, Dict, Any, Optional
from kaggle_secrets import UserSecretsClient

class KaggleInfra:
    @staticmethod
    def load_secrets():
        user_secrets = UserSecretsClient()
        return {
            "url": user_secrets.get_secret("FIREBASE_URL"),
            "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),
            "telegram": user_secrets.get_secret("TELEGRAM_TOKEN")
        }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred, {'databaseURL': secrets['url']})
        self.root_ref = db.reference("glu_stock")

    def push_task(self, queue_name: str, data: Any):
        self.root_ref.child(f"task_queue/{queue_name}").push(data)
        
    def log_event(self, phase, details):
        self.root_ref.child("history").push({
            'timestamp': datetime.now().isoformat(),
            'phase': phase.upper(),
            'details': details
        })

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Research Agent)
import yfinance as yf
import pandas as pd

class ResearchAgent:
    def get_market_data(self, ticker: str, period: str = "1y") -> pd.DataFrame:
        data = yf.download(ticker, period=period, interval="1d", progress=False)
        return data

    def fundamental_filter(self, ticker: str) -> bool:
        try:
            info = yf.Ticker(ticker).info
            # Filter logic: Growth + Valuation
            growth = info.get('earningsQuarterlyGrowth', 0) or 0
            pe = info.get('trailingPE', 100)
            return growth > 0.05 and pe < 25
        except: return False

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_scan():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    research = ResearchAgent()
    
    universe = ["BBCA.JK", "TLKM.JK", "ASII.JK", "UNTR.JK", "ADRO.JK", "BMRI.JK", "ARTO.JK", "GOTO.JK"] 
    candidates = []
    
    print(f"🔭 Scanning {len(universe)} stocks...")
    for ticker in universe:
        if research.fundamental_filter(ticker):
            candidates.append(ticker)
            print(f"✅ {ticker} passed fundamental filter.")
            
    if candidates:
        fb.push_task("research", candidates)
        fb.log_event("RESEARCH", f"Pushed {len(candidates)} stocks to queue.")
        print(f"🚀 Successfully pushed analysis tasks for {candidates}")
    else:
        print("💤 No opportunities found today.")

run_scan()